# **TASK 0: The Library of Babel**

---

A pretty cool assignment.

I will try to add self-explanatory comments about some code snippets in order to communicate my thought process and the rationale behind the decisions I will make.

<br>

**[Edit]:** 

- Please run the cells in order if you want to reproduce my work, however, I have run all cells in several iterations to make sure that you get to look at the outpurs and logs without having to run the code yourself. 

- This "cool" task took up a lot of my time but it was fun to work on. Thanks for the opportunity, Prof. PK and team Precog! :)

### **Imports and Setup**

In [9]:
import json
import re
import random
from pathlib import Path
import pandas as pd
import spacy


RANDOM_SEED = 42 # Setting it here to ensure reproducibility throughout the notebook
random.seed(RANDOM_SEED)

nlp = spacy.load("en_core_web_sm") # Loading the small SpaCy model for sentence segmentation

### **Author selection rationale**

**1. Mary Shelley - Frankenstein**

While skimming through the novel at a superficial level and reading summaries online, I found that readers [[1]](https://rajuel.blogspot.com/2019/02/frankestein-epistolary-form-of-writing.html) [[2]](https://www.enotes.com/topics/frankenstein/questions/what-is-mary-shelley-s-writing-style-in-190177) describe her writing style as "Long epistolary philosophical with a Gothic tone". I myself saw that she uses complex nested sentences and a very formal Victorian vocabulary.

**2. Charles Dickens - Great Expectations**

This was an easier read (at a superficial level, of course) as compared to Frankenstein because Dickens used shorter satirical sentences and a more conversational tone. While going through the internet articles and blogs [[3]](https://www.bbc.co.uk/bitesize/guides/zg9wnbk/revision/3) about this novel, I got to know that Dickens used a technique called "onomatopoeia" wherein the sounds of the names of his characters give us an idea about their personality. Thus, his tone is much more direct as compared to the philosophical tone of Mary Shelley.


**Conclusion**

Both the authors belong to the Victorian era (1837-1901) which controls for the time period, and they write about similar themes of society, morality, and human nature. However, their writing styles are quite different in these novels. I did this to ensure that the classification signals styles rather than the topics.

(Thanks to my sister for helping me with these choices)



### **[Edit]**
**3. Mathilda by Mary Shelley**

Initial segmentation of Frankenstein produced only 376 segments after filtering, falling short of the 500-per-author target. To address this while maintaining class homogeneity, I am adding "Mathilda" by Mary Shelley. This will be pooled with Frankenstein fulfil the 500 segment requirements for Mary Shelley in Class 1. 

PS. "Great Expectations" already had more than 500 paragraphs after filtering, so no additional text was needed for Charles Dickens.

**Why Mathilda?**

Same author, same voice as Frankenstein, but a different story. Sufficient length to yield the required number of segments after filtering. Also, there is a good theme overlap: Gothic, isolation, rejection.


### **Cleaning the novels** 

Chapter headings and Roman numeral headers were removed using a regex pattern while extra spaces and newlines were cleaned up using simple string manipulation.
Also, there were blank lines in the novels which had spaces or tabs, so I used a regex pattern to remove those as well.

I normalized the line endings from carriage return + newline (`\r\n`) to just newline (`\n`) and stripped trailing whitespace from each line to preserve the structure of the text as much as possible.

It was quite easy to remove the Gutenberg headers and footers, as they are consistent across all novels. Since there were only two novels, I was skimming through them to figure out some good topics for class 2 and class 3 datasets. Thus, I found and removed the unnecessary text (above the header and below the footer) while doing that. 

In Great Expectations, I noticed that there were illustration placeholders (`[Illustration]`) that were not part of the actual text, so I removed those as well.

I tried writing a script to safely remove the contents list but it was a bit tricky to do it in a way that would work for both novels, so I ended up removing it manually since it is at the beginning of the novels and does not affect the structure of the text.

### **[Edit]**

**Cleaning Mathilda**

Unlike Frankenstein and Great Expectations, which follow standard Project Gutenberg formatting, Mathilda includes non-standard elements: a preface by the editor, introductory remarks, a contents list, and footnotes spread across the text.
Thus, I had to manually remove these because they are editorial metadata. It took me around 10 minutes to do so, which is significantly faster and more accurate than writing specific regex patterns to handle all the edge-cases.

However, the manually cleaned Mathilda text underwent identical cleaning steps as the other two novels to ensure consistency in formatting and structure across all three texts.

In [40]:
def clean_novel_text(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8-sig', errors='replace') as f:
        text = f.read()

    original_char_count = len(text)
    original_word_count = len(text.split())

    print(f"\nCleaning: {Path(input_path).name} ")
    print(f"Original Character Count: {original_char_count:,}")
    print(f"Original Word Count (approx): {original_word_count:,}")

    # Normalizing line endings
    text = text.replace('\r\n', '\n')
    print("Normalized line endings.")

    # Removing chapter and volume headings using regex patterns
    pattern = r'^\s*(?:CHAPTER|Chapter|VOLUME|Volume)\s*[IVivXL\d.]*\s*$\n?'
    text = re.sub(pattern, '', text, flags=re.MULTILINE)
    print("Removed chapter and volume headings.")

    # Using regex to remove illustration placeholders
    text = re.sub(r'\[Illustration[^\]]*\]', '', text)
    print("Removed illustration placeholders.")

    # Stripping trailing whitespace from each line (trying to preserve structure)
    text = '\n'.join(line.rstrip() for line in text.split('\n'))
    print("Stripped trailing whitespace from each line.")

    # Removing excessive blank lines (> 2 consecutive) and blank lines with only spaces/tabs
    text = re.sub(r'\n[ \t]+\n', '\n\n', text)
    text = re.sub(r'\n\n\n+', '\n\n', text)
    print("Removed excessive blank lines.")

    # Stripping leading and trailing whitespace from the entire text
    text = text.strip()

    cleaned_char_count = len(text)
    cleaned_word_count = len(text.split())

    print(f"Cleaned Character Count: {cleaned_char_count:,}")
    print(f"Cleaned Word Count (approx): {cleaned_word_count:,}")
    print(f"Characters removed: {original_char_count - cleaned_char_count:,} ({100*(original_char_count - cleaned_char_count)/original_char_count:.1f}%)")

    # Sampling text from the middle just for visual verification
    middle_idx = len(text) // 2
    sample_start = max(0, middle_idx - 250)
    sample_end = min(len(text), middle_idx + 250)
    sample = text[sample_start:sample_end]    

    print(f"\n --- Sample from the middle (500 chars) ---\n")
    print(sample)
    print(f"\n --- End of Sample ---\n")

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(text)
    print(f"Cleaned text saved to: {output_path}")

    return {
        'original_chars': original_char_count,
        'original_words': original_word_count,
        'cleaned_chars': cleaned_char_count,
        'cleaned_words': cleaned_word_count,
        'chars_removed': original_char_count - cleaned_char_count,
        'percent_removed': 100 * (original_char_count - cleaned_char_count) / original_char_count
    }

In [41]:
# Just calling the functions and printing some stats (EDIT: I had to add in Mathilda)

frank_stats = clean_novel_text('../data/raw/Frankenstein.txt', '../data/processed/frankenstein_cleaned.txt')

dickens_stats = clean_novel_text('../data/raw/Great Expectations.txt', '../data/processed/great_expectations_cleaned.txt')

mathilda_stats = clean_novel_text('../data/raw/Mathilda.txt', '../data/processed/mathilda_cleaned.txt')

print("\nCleaning Summary:")
print("-" * 30)
summary_df = pd.DataFrame({
    'Novel': ['Frankenstein (Shelley)', 'Great Expectations (Dickens)', 'Mathilda (Shelley)'],
    'Original Chars': [f"{frank_stats['original_chars']:,}", f"{dickens_stats['original_chars']:,}", f"{mathilda_stats['original_chars']:,}"],
    'Cleaned Chars': [f"{frank_stats['cleaned_chars']:,}", f"{dickens_stats['cleaned_chars']:,}", f"{mathilda_stats['cleaned_chars']:,}"],
    'Chars Removed': [f"{frank_stats['chars_removed']:,}", f"{dickens_stats['chars_removed']:,}", f"{mathilda_stats['chars_removed']:,}"],
    '% Removed': [f"{frank_stats['percent_removed']:.2f}%", f"{dickens_stats['percent_removed']:.2f}%", f"{mathilda_stats['percent_removed']:.2f}%"]
})
print(summary_df.to_string(index=False))


Cleaning: Frankenstein.txt 
Original Character Count: 418,796
Original Word Count (approx): 74,955
Normalized line endings.
Removed chapter and volume headings.
Removed illustration placeholders.
Stripped trailing whitespace from each line.
Removed excessive blank lines.
Cleaned Character Count: 418,349
Cleaned Word Count (approx): 74,907
Characters removed: 447 (0.1%)

 --- Sample from the middle (500 chars) ---

view of the several empires at present existing in the world; it gave
me an insight into the manners, governments, and religions of the different
nations of the earth. I heard of the slothful Asiatics, of the stupendous
genius and mental activity of the Grecians, of the wars and wonderful virtue
of the early Romans—of their subsequent degenerating—of the
decline of that mighty empire, of chivalry, Christianity, and kings. I heard
of the discovery of the American hemisphere and wept with Safie ov

 --- End of Sample ---

Cleaned text saved to: ../data/processed/frankenstein_c

<a id = "topic-extraction"></a>
### **Topic Extraction**

I skimmed through both novels (mostly the opening and closing chapters) and cross-referenced with a few online theme summaries [[1]](https://www.litcharts.com/lit/frankenstein/themes) [[2]](https://www.sparknotes.com/lit/frankenstein/themes/) [[3]](https://www.reddit.com/r/literature/comments/1g4zlek/common_interpretation_of_frankenstein/) [[4]](https://www.sparknotes.com/lit/greatex/themes/) [[5]](https://www.litcharts.com/lit/great-expectations/themes) [[6]](https://www.enotes.com/topics/great-expectations/in-depth/style-form-literary-elements/themes) [[7]](https://www.sparknotes.com/lit/matilda/themes/) to settle on the following. I tried to keep the themes broad enough to be generatable by Gemini API while still being grounded in the actual text.

PS. I tried to take help from Gemini and ChatGPT first but they kept giving me textbook-level summaries, so I ended up doing it manually using the cited resources and some help from my sister again.

**Mary Shelley** *(Frankenstein and Mathilda)*

1. *The hubris of scientific creation*

2. *The isolation and rejection of the outcast*

3. *The ethics of love, parenthood and abandonment*

4. *Grief and the impossibility of recovery*

5. *Nature and the sublime as emotional mirror*

**Charles Dickens** *(Great Expectations)*

1. *Social class and the illusion of gentility*

2. *Wealth, ambition, and moral corruption*

3. *Crime, guilt, and redemption*

4. *Self-deception and identity*

5. *The past as a trap*

**NOTE:** A few themes overlap loosely across both authors: isolation, guilt and the weight of the past. It means the AI-generated paragraphs for Class 2 and Class 3 will have coverage across all 10 topics without forcing artificial separations, and any classification signal will more cleanly reflect style rather than topic. (Hopefully)

In [13]:
# Addings these topics to a json file for future use in Class 2 and Class 3 datasets.

topics = [
    "The hubris of scientific creation",
    "The isolation and rejection of the outcast",
    "The ethics of love, parenthood, and abandonment",
    "Grief and the impossibility of recovery",
    "Nature and the sublime as emotional mirror",
    "Social class and the illusion of gentility",
    "Wealth, ambition, and moral corruption",
    "Crime, guilt, and redemption",
    "Self-deception and identity",
    "The past as a trap"
]

topics_dict = {"topics": topics}
with open('../data/topics.json', 'w') as f:
    json.dump(topics_dict, f, indent=4)
print("\nSaved topics to ../data/topics.json")


Saved topics to ../data/topics.json


### **Text Segmentation**

The aim was to split the cleaned text into segments of 150-200 words with proper sentence boundaries. This was necessary because the Class 2 and Class 3 datasets require AI-generated paragraphs of 100-200 words each, and there should be semantic coherence within each segment.
My first attempt failed miserably.

**First Attempt (Regex-based sliding window):**

I accumulated words until ≥150, then kept adding until word count hit max (200) or a regex-detected sentence boundary.

- **Problem:** Regex patterns like `[.!?]+$` only detected *punctuation position* in words instead of actual sentence structure. For example, it detected the period in "Dr." as a sentence boundary.
- **Result:** Despite visible sentence markers, segments frequently began or ended mid-clause. 
- **Why I think it failed:** Regex operates on character position alone, and punctuation-at-end detection is unreliable for such complex Victorian proses.

**Second Attempt (SpaCy NLP-based):**

I used SpaCy's `doc.sents` to identify true sentence boundaries, then grouped complete sentences within the 150-200 word range. At the end, I got 945 paragraphs for "Great Expectations", 396 for "Frankenstein", and 160 for "Mathilda". 

Since Frankenstein yielded only 396 paragraphs, I had the option to either relax the word count constraints or add another novel by Mary Shelley to meet the 500-paragraph target. I chose the latter and added "Mathilda" by Mary Shelley. 

I loaded 556 paragraphs from "Mathilda" and "Frankenstein" combined, and then randomly sampled 500 paragraphs from this pool to create the Class 1 dataset for Mary Shelley. 

Finally, 1000 paragraphs (500 Shelley + 500 Dickens) were saved as JSONL with metadata `{text, author, label=0}` for Class 1.

In [42]:
# We are extracting all sentences using SpaCy's sentence tokenizer. Then we keep adding sentences while word_count < max_words. 
# Once the word_count exceeds max_words AND adding the next sentence would exceed max_words, we close the segment (paragraph).

def sentence_boundary(cleaned_text, min_words=150, max_words=200):
    doc = nlp(cleaned_text)
    sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]
    if not sentences:
        return []
    
    segments = []
    current_sentences = []
    current_word_count = 0

    for sent in sentences:
        sent_words = len(sent.split())
        
        # If current segment has min_words and adding this sentence would exceed max_words, save current segment and start new
        if current_word_count >= min_words and current_word_count + sent_words > max_words:
            segments.append(' '.join(current_sentences))
            current_sentences = [sent]
            current_word_count = sent_words
        else:
            # Adding the complete sentence to current segment
            current_sentences.append(sent)
            current_word_count += sent_words
    
    # Saving the final segment if it meets minimum word count
    if current_word_count >= min_words and current_sentences:
        segments.append(' '.join(current_sentences))

    return segments

In [50]:
# We are segmenting the cleaned novel text using the sentence_boundary function above and then filtering segments based on word count for consistency in training data for future use.
def segment_and_filter(cleaned_text_path, output_jsonl_path):
    with open(cleaned_text_path, 'r', encoding='utf-8') as f:
        text = f.read()

    raw_segments = sentence_boundary(text)
    print(f"Segments (sentence-bounded): {len(raw_segments)}")

    word_counts = {}
    for i, seg in enumerate(raw_segments):
        word_counts[i] = len(seg.split())

    # Filtering segments based on word count (150-200 words) because Class 2 and 3 datasets will need paragraphs of similar length.
    filtered_indices = [i for i, wc in word_counts.items() if 150 <= wc <= 200]
    filtered_segments = [raw_segments[i] for i in filtered_indices]
    print(f"Segments after filtering (150-200 words): {len(filtered_segments)}")

    filtered_word_counts = [word_counts[i] for i in filtered_indices]
    if filtered_word_counts:
        min_wc = min(filtered_word_counts)
        max_wc = max(filtered_word_counts)
        mean_wc = sum(filtered_word_counts) / len(filtered_word_counts)
        median_wc = sorted(filtered_word_counts)[len(filtered_word_counts) // 2]

        print(f"Word Count Stats for Filtered Segments:")
        print(f"  Min: {min_wc} words")
        print(f"  Max: {max_wc} words")
        print(f"  Mean: {mean_wc:.2f} words")
        print(f"  Median: {median_wc} words")
    else:
        min_wc = max_wc = mean_wc = median_wc = 0

    # Shuffling with reproducible seed (RANDOM_SEED is set at the top of the notebook)
    random.shuffle(filtered_segments)

    # Taking first 500 segments (or all if less than 500) 
    final_segments = filtered_segments[:500]
    actual_count = len(final_segments)
    if actual_count < 500:
        print(f"Only {actual_count} segments available after filtering.")
    else:
        print(f"Selected first 500 segments")

    with open(output_jsonl_path, 'w') as f:
        for text_segment in final_segments:
            # Calculating actual word count from the text
            wc = len(text_segment.split())
            record = {
                'text': text_segment,
                'topic': None, # Placeholder for future use in Class 2 and Class 3
                'label': 0, # Label 0 for Class 1 dataset
                'word_count': wc # Adding actual word count to the record for future analysis 
            }
            f.write(json.dumps(record) + '\n')
    print(f"Saved {actual_count} segments to: {output_jsonl_path}")
    
    return{
        'raw_count': len(raw_segments),
        'after_filter': len(filtered_segments),
        'final_count': actual_count,
        'min_words': min_wc,
        'max_words': max_wc,
        'mean_words': mean_wc
    }

In [51]:
# Calling the above function for the novels

frankenstein_seg_stats = segment_and_filter('../data/processed/frankenstein_cleaned.txt', '../data/class1/extras/class1_frankenstein.jsonl')

dickens_seg_stats = segment_and_filter('../data/processed/great_expectations_cleaned.txt', '../data/class1/class1_dickens.jsonl')

mathilda_seg_stats = segment_and_filter('../data/processed/mathilda_cleaned.txt', '../data/class1/extras/class1_mathilda.jsonl')

Segments (sentence-bounded): 402
Segments after filtering (150-200 words): 396
Word Count Stats for Filtered Segments:
  Min: 150 words
  Max: 200 words
  Mean: 185.67 words
  Median: 188 words
Only 396 segments available after filtering.
Saved 396 segments to: ../data/class1/extras/class1_frankenstein.jsonl
Segments (sentence-bounded): 985
Segments after filtering (150-200 words): 945
Word Count Stats for Filtered Segments:
  Min: 150 words
  Max: 200 words
  Mean: 185.61 words
  Median: 188 words
Selected first 500 segments
Saved 500 segments to: ../data/class1/class1_dickens.jsonl
Segments (sentence-bounded): 178
Segments after filtering (150-200 words): 160
Word Count Stats for Filtered Segments:
  Min: 150 words
  Max: 200 words
  Mean: 183.72 words
  Median: 186 words
Only 160 segments available after filtering.
Saved 160 segments to: ../data/class1/extras/class1_mathilda.jsonl


### **Pooling Frankenstein and Mathilda**

I am pooling 'Frankenstein' (396 paragraphs) and 'Mathilda' (160 paragraphs) and selecting 500 randomly sampled paragraphs to create a larger dataset for Mary Shelley. I have also selected the first 500 paragraphs from the randomly shuffled 'Great Expectations' to create the dataset for Charles Dickens. 

**An interesting observation** is that in approach 1 of text segmentation, I got a mean of *199 words per paragraph* because it was a strict (not so accurate) sliding window with a regex-based sentence boundary detection. In approach 2, I am getting a mean of around *185 words per paragraph* in all three novels because it is based on actual sentence boundaries detected by SpaCy, which allows for more natural paragraph breaks.

<br>
PS. I accidentally deleted my approach 1 code cell, otherwise it would have been a good inclusion in this notebook.

In [45]:
# Pooling Mary Shelley segments from Frankenstein and Mathilda

print("SHELLEY CLASS 1 POOLING: Frankenstein + Mathilda")
print(f"{'-'*70}")

def load_jsonl(filepath):
    records = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

# Loading both pre-segmented Mary Shelley JSONL files
frankenstein = load_jsonl('../data/class1/extras/class1_frankenstein.jsonl')
mathilda = load_jsonl('../data/class1/extras/class1_mathilda.jsonl')

print(f"Frankenstein segments: {len(frankenstein)}")
print(f"Mathilda segments: {len(mathilda)}")

# Pooling both texts
pooled = frankenstein + mathilda
print(f"Total Shelley (before selection): {len(pooled)}")

# Shuffling for random distribution while maintaining reproducibility
random.shuffle(pooled)

# Taking first 500 segments
final_segments = pooled[:500]
actual_count = len(final_segments)

print(f"Final selection: first {actual_count} segments after shuffle")

with open('../data/class1/class1_shelley.jsonl', 'w', encoding='utf-8') as f:
    for segment in final_segments:
        wc = len(segment['text'].split())
        record = {
            'text': segment['text'],
            'topic': None, 
            'label': 0,  # Class 1
            'word_count': wc  # Adding actual word count to the record for future analysis
        }
        f.write(json.dumps(record) + '\n')

print(f"Saved {actual_count} pooled Shelley segments to: ../data/class1/class1_shelley.jsonl")

SHELLEY CLASS 1 POOLING: Frankenstein + Mathilda
----------------------------------------------------------------------
Frankenstein segments: 396
Mathilda segments: 160
Total Shelley (before selection): 556
Final selection: first 500 segments after shuffle
Saved 500 pooled Shelley segments to: ../data/class1/class1_shelley.jsonl


In [47]:
# Validation (Inspecting a sample from the JSONL files)

def load_jsonl(filepath):
    records = []
    with open(filepath, 'r') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

shelley_data = load_jsonl('../data/class1/class1_shelley.jsonl')
dickens_data = load_jsonl('../data/class1/class1_dickens.jsonl')

print(f"\nSample from Class 1 - Shelley:")
print("-"*70)
sample_shelley = shelley_data[40] # Random index for sampling
print(f"Label: {sample_shelley['label']}") # Should be 0 for Class 1 dataset
print(f"Word Count: {len(sample_shelley['text'].split())} words")
print(f"Text:\n"f"{sample_shelley['text'][:500]}...") # Printing the first 500 chars

print(f"\nSample from Class 1 - Dickens:")
print("-"*70)
sample_dickens = dickens_data[40] 
print(f"Label: {sample_dickens['label']}")
print(f"Word Count: {len(sample_dickens['text'].split())} words")
print(f"Text:\n{sample_dickens['text'][:500]}...")


Sample from Class 1 - Shelley:
----------------------------------------------------------------------
Label: 0
Word Count: 200 words
Text:
The misfortunes of Woodville were not of the hearts core like
mine; his was a natural grief, not to destroy but to purify the heart
and from which he might, when its shadow had passed from over him,
shine forth brighter and happier than before. Woodville was the son of a poor clergyman and had received a classical
education. He was one of those very few whom fortune favours from
their birth; on whom she bestows all gifts of intellect and person
with a profusion that knew no bounds, and whom unde...

Sample from Class 1 - Dickens:
----------------------------------------------------------------------
Label: 0
Word Count: 181 words
Text:
We got ashore among some slippery
stones while we ate and drank what we had with us, and looked about. It
was like my own marsh country, flat and monotonous, and with a dim
horizon; while the winding river turned and

### **Class 2 and Class 3 Dataset Generation**

Since I am done with Class 1 dataset generation, I am moving on to Class 2 and Class 3. The plan is to prompt Gemini API to generate 50 paragraphs for each of the 10 topics (5 per author), which will give us 500 paragraphs for Class 2 and 500 paragraphs for Class 3. 

This will require a huge number of API calls and hence, I have chosen to use `Gemini 3.1 Flash Lite` model because it offers a rate limit of 500 requests per day (RPD), which is the highest among all the free-tier Gemini models.
To be on the safer side, I have created 4 API keys from different Google accounts which can be rotated in case I hit the RPD limit on one of them.

I will be setting the **temperature at 0.9 for Class 2** generation and **at 0.7 for Class 3**. This is because the former is supposed to have more lexical diversity as compared to the latter, which is supposed to be more specific to the instructed writing style (Mary Shelley). The prompt will instruct Gemini to generate paragraphs of 150-180 words as an attempt to be consistent with the Class 1 dataset's mean of around 185 words per paragraph.

Checkpointing is a part of my plan because I am expecting to hit some errors mid-run and I don't want to lose progress. The logic is simple: after each successful API call, I will get a batch (5) of paragraphs, and will append them to my checkpoint (the ones which clear the word count filter).

<br>

**[Edit]**: This step took some serious turns. I have added some self-explanatory in-line comments in the code cells. Hope they help!

In [2]:
# Import statements for Class 2 and Class 3 Dataset generation
# I am re-importing some from the top of the notebook, just in case the session times out while reproducing this notebook.

import google.genai as genai
from pathlib import Path
import json
import time
import random
import re
import os
from dotenv import load_dotenv

In [ ]:
# Configurations

load_dotenv('../.env') # Considering that the .env file is in the root directory
API_KEY = os.getenv('api_key', '') # I have stored multiple API keys as comma-separated values in the "api_key" variable in the .env file
if not API_KEY:
    raise ValueError("api_key not found in the .env file.")

API_KEYS = [key.strip() for key in API_KEY.split(',') if key.strip()] # 
print(f"Loaded {len(API_KEYS)} API keys from .env file.")

with open('../data/topics.json', 'r') as f: # Loading the topics that I earlier extracted from the 3 novels
    topics_data = json.load(f)
TOPICS = topics_data.get('topics', [])
print(f"Loaded {len(TOPICS)} topics.")

TARGET_PER_TOPIC = 50       # 10 topics × 50 paragraphs per topic = 500 total paragraphs per class
BATCH_SIZE = 5              # paragraphs per API call
# I am keeping MIN_WORDS and MAX_WORDS a bit wider here because Gemini might undercount or overcount words slightly.
MIN_WORDS = 130 
MAX_WORDS = 200 
SLEEP_BETWEEN_CALLS = 5    # 5 seconds gap between consecutive API calls to stay under 15 RPM rate limit
RANDOM_SEED = 42 # Re-setting it here for reproducibility 

MODEL_NAME = "gemini-3.1-flash-lite-preview" # Using this model because it is the Gemini model with highest requests per day (RPD) limit
# Temperature is a measure of randomness in the output because it controls the probability distribution of the next word prediction. 
TEMPERATURE_CLASS2 = 0.9 # I have made this arbitrary choice to have a higher temperature for Class 2 to maximise lexical variety across the generic paragraphs
TEMPERATURE_CLASS3 = 0.7 # Using a slightly lower temperature for Class 3 because we want it to be more focused on mimicking Mary Shelley's style 

CHECKPOINT_DIR = Path("../data/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

Loaded 3 API keys from .env file.
Loaded 10 topics.


In [4]:
# An API Key rotator class built by Copilot. (I am bad at Object Oriented Programming)
# The idea here is to have a class that manages multiple API keys and rotates to the next key whenever we hit a rate limit (429 error). 
class KeyRotator:
    def __init__(self, keys):
        self.keys = keys
        self.idx = 0

    def current(self):
        return self.keys[self.idx]

    def rotate(self):
        self.idx = (self.idx + 1) % len(self.keys)
        print(f"  → Rotated to key index {self.idx}")
        time.sleep(5)

    def get_client(self):
        return genai.Client(api_key=self.current())


In [5]:
# Prompt templates for the Gemini API
# I am hoping that these prompts will work.

def prompt_class2(topic, n):
    return f"""Write exactly {n} separate prose paragraphs on the following topic: "{topic}"

Requirements:
- STRICT: Each paragraph must be between 150 and 180 words. Count carefully before outputting.
- Write in neutral, clear, modern English prose. No lists, no headers, no bullet points.
- Do not imitate any specific author or literary style.
- Do not begin paragraphs with a topic sentence that reads like an essay introduction. Start mid-thought if needed.
- Each paragraph must stand alone as a complete, coherent piece of writing.
- Separate each paragraph from the next with exactly one blank line.
- Do not number the paragraphs. Do not add titles.
- Do not include any preamble or closing remarks. Output only the paragraphs.
"""

def prompt_class3(topic, n, author="Mary Shelley"):
    return f"""Write exactly {n} separate prose paragraphs on the following topic: "{topic}"

Requirements:
- STRICT: Each paragraph must be between 150 and 180 words. Count carefully before outputting.
- Write in the style of {author}. Specifically:
    * Use long, complex sentences with nested subordinate clauses.
    * Use formal Victorian vocabulary and a philosophical, introspective tone.
    * Employ first-person narration where natural, with epistolary emotional weight.
    * Use the Gothic register: nature as emotional mirror, the sublime, moral weight.
    * Avoid modern phrasing, slang, or casual constructions entirely.
- Each paragraph must stand alone as a complete, coherent piece of writing.
- Separate each paragraph from the next with exactly one blank line.
- Do not number the paragraphs. Do not add titles.
- Do not include any preamble or closing remarks. Output only the paragraphs.
"""

In [ ]:
# Helping functions (Parsing, Validating, and Checkpointing)

# I will split the response on blank lines because I have promopted Gemini to separate paragraphs with one blank line
def parse_response(response_text):
    blocks = re.split(r'\n\s*\n', response_text.strip())
    return [b.strip() for b in blocks if b.strip()]

def word_count(text): # It is annoying to repeatedly calculate word count
    return len(text.split())

# Checking if the word count requirements are met
def validate_paragraphs(paragraphs, min_w=MIN_WORDS, max_w=MAX_WORDS):
    valid = []
    rejected = [] # I am maintaining this rejected list just to keep track of how many paragraphs are being rejected 
    for p in paragraphs:
        wc = word_count(p)
        if min_w <= wc <= max_w:
            valid.append(p)
        else:
            rejected.append((wc, p[:60]))
    return valid, rejected

# Checkpointing functions to save progress in case of any interruption
def checkpoint_path(class_name, topic_idx):
    safe_topic = re.sub(r'[^\w]', '_', TOPICS[topic_idx])[:40] # Making a safe filename by replacing non-alphanumeric characters with underscores and truncating to 40 chars
    return CHECKPOINT_DIR / f"{class_name}_topic{topic_idx:02d}_{safe_topic}.json"

def load_checkpoint(class_name, topic_idx): 
    path = checkpoint_path(class_name, topic_idx)
    if path.exists():
        with open(path) as f:
            data = json.load(f)
        print(f"Checkpoint found: {len(data)} paragraphs already collected")
        return data
    return []

def save_checkpoint(class_name, topic_idx, paragraphs):
    path = checkpoint_path(class_name, topic_idx)
    with open(path, 'w') as f:
        json.dump(paragraphs, f)

In [ ]:
# Core generation functions

# Generating target number of paragraphs for a given topic (in my case, it is 50 paragraphs per topic)
def generate_for_topic(topic, topic_idx, class_name, prompt_fn, temperature, rotator):
    collected = load_checkpoint(class_name, topic_idx) # Loading any previously collected paragraphs for the current topic

    attempts = 0
    max_attempts = 20  # Safety ceiling per topic

    while len(collected) < TARGET_PER_TOPIC and attempts < max_attempts: # Looping until we have enough paragraphs or we hit max attempts
        still_needed = TARGET_PER_TOPIC - len(collected) # Self explanatory (how many more paragraphs to be generated for the current topic)
        batch = min(BATCH_SIZE, still_needed + 3)  # I am adding this 3 as buffer just in case some paragraphs get rejected for word count

        prompt = prompt_fn(topic, batch)

        # Copilot suggested the API key rotation logic entirely. I came up with the idea of looking for "429" or "quota" or "rate" in the error message
        try:    
            client = rotator.get_client()
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config={"temperature": temperature}
            )
            # Extracting text from the response object
            raw_text = response.text if response.text else ""
            if not raw_text: # If the response text is empty (maybe due to the safety filter blocking the crime and guilt content), we will skip this batch and try again
                print(f"  Empty response received (possibly the safety filter). Skipping the batch.")
                attempts += 1
                time.sleep(SLEEP_BETWEEN_CALLS)
                continue
            
        except Exception as e:
            err = str(e)
            if "429" in err or "quota" in err.lower() or "rate" in err.lower(): 
                print(f"Rate limit hit. Rotating the API key...") 
                rotator.rotate() 
                time.sleep(30)
                continue
            else:
                print(f"API error: {err}. Sleeping 30s then retrying...")
                time.sleep(30)
                attempts += 1
                continue

        parsed = parse_response(raw_text)
        valid, rejected = validate_paragraphs(parsed)
        collected.extend(valid[:still_needed]) # Adding only the valid paragraphs that we still need to reach the target count
        save_checkpoint(class_name, topic_idx, collected) 

        print(f"Batch {attempts + 1}: {len(valid)} valid, {len(rejected)} rejected. "
              f"Total: {len(collected)}/{TARGET_PER_TOPIC}")

        if rejected:
            print(f"Rejected word counts: {[r[0] for r in rejected]}") # Printing the rejected paragraphs' word counts for debugging

        attempts += 1
        time.sleep(SLEEP_BETWEEN_CALLS) # Sleeping between API calls to respect rate limits

    # A warning message in case we exit the loop without reaching the target count for the topic
    if len(collected) < TARGET_PER_TOPIC:
        print(f"WARNING: Only got {len(collected)}/{TARGET_PER_TOPIC} paragraphs for topic '{topic}'")

    return collected[:TARGET_PER_TOPIC]

# The main function to generate the dataset for either of the classes 
def generate_class(class_name, prompt_fn, temperature, label, output_path):
    rotator = KeyRotator(API_KEYS) 

    all_records = [] # This will hold all the paragraphs for the entire class (all topics combined) before we shuffle and save to JSONL
    per_topic_counts = {} #This will keep track of how many paragraphs we collected for each topic (only for logging purposes)

    print(f"GENERATING {class_name.upper()}: {label}")
    print(f"{'-'*70}")

    for topic_idx, topic in enumerate(TOPICS): # Looping through each topic in the TOPICS list
        print(f"\n[{topic_idx + 1}/{len(TOPICS)}] {topic}")
        paragraphs = generate_for_topic(
            topic, topic_idx, class_name, prompt_fn, temperature, rotator
        )

        per_topic_counts[topic] = len(paragraphs)

        for para in paragraphs:
            all_records.append({
                "text": para,
                "topic": topic,
                "label": label,
                "word_count": word_count(para)
            })

    # Shuffling the final dataset because unshuffled data might lead to topic-wise clustering in the training process later on
    print(f"\nShuffling {len(all_records)} records...")
    random.seed(RANDOM_SEED) # Already set as 42 
    random.shuffle(all_records)

    with open(output_path, 'w', encoding='utf-8') as f:
        for record in all_records:
            f.write(json.dumps(record) + '\n')

    print(f"\n{'-'*70}")
    print(f"SAVED: {output_path}")
    print(f"Total records: {len(all_records)}")

    # Some summary stats
    word_counts = [r["word_count"] for r in all_records]
    print(f"\nWord count distribution:")
    print(f"  Min: {min(word_counts)}, Max: {max(word_counts)}, Mean: {sum(word_counts)/len(word_counts):.1f}")

    # Per-topic summary stats
    print(f"\nPer-topic record counts:")
    print(f"{'-'*70}")
    for topic in TOPICS:
        count = per_topic_counts.get(topic, 0)
        status = "Done" if count == TARGET_PER_TOPIC else f"Not Done({count})" # Marking topics that did not reach the target count
        print(f"  {status:12s}  {count:3d}/{TARGET_PER_TOPIC}  {topic}")

    # Checking if we reached the overall target count for the entire class (all topics combined)
    total_collected = sum(per_topic_counts.values()) 
    if total_collected == (len(TOPICS) * TARGET_PER_TOPIC):
        print(f"COMPLETE: All {len(TOPICS)} topics * {TARGET_PER_TOPIC} = {total_collected} records")
    else:
        print(f"INCOMPLETE: Only {total_collected}/{len(TOPICS) * TARGET_PER_TOPIC} records collected")

    print(f"{'-'*70}\n")

    return all_records

In [ ]:
# Calling the generation function for both the classes
# Just to be on the safer side, I am generating Class 2 first and then doing a manual inspection before proceeding to Class 3 generation.

print("\nSTARTING CLASS 2 GENERATION")

records_class2 = generate_class(
    class_name="class2",
    prompt_fn=prompt_class2,
    temperature=TEMPERATURE_CLASS2,
    label=1, # Label 0 for Class 1, Label 1 for Class 2, Label 2 for Class 3
    output_path="../data/class2/class2_generic_ai.jsonl"
)

# Manual inspection step
print("\n" + "-"*70)
print("MANUAL INSPECTION: Class 2")
print("\nShowing 3 sample paragraphs from the first topic:")
print("-"*70)

# I am showing the first 3 paragraphs from the first topic just to get a sense of the quality of the generated text
first_topic_samples = [r for r in records_class2 if r["topic"] == TOPICS[0]][:3]
for i, sample in enumerate(first_topic_samples, 1):
    print(f"\n[Sample {i}] ({sample['word_count']} words):")
    print(sample['text'][:300] + "..." if len(sample['text']) > 300 else sample['text'])

user_input = input("\nType 'yes' to proceed to Class 3, or 'no' to stop: ").strip().lower() 
if user_input != 'yes':
    print("Stopping. Edit the Class 2 prompt and re-run.")
    raise SystemExit

# Beginning Class 3 generation now, if Class 2 looks good
print("\nSTARTING CLASS 3 GENERATION")

records_class3 = generate_class(
    class_name="class3",
    prompt_fn=prompt_class3,
    temperature=TEMPERATURE_CLASS3,
    label=2,
    output_path="../data/class3/class3_mimic_ai.jsonl"
)

print("\n" + "-"*70)
print("CLASS 2 & CLASS 3 GENERATION COMPLETE")
print(f"Class 2: {len(records_class2)} records at ../data/class2/class2_generic_ai.jsonl")
print(f"Class 3: {len(records_class3)} records at ../data/class3/class3_mimic_ai.jsonl")
print("-"*70 + "\n")


STARTING CLASS 2 GENERATION
GENERATING CLASS2: 1
----------------------------------------------------------------------

[1/10] The hubris of scientific creation
Batch 1: 5 valid, 0 rejected. Total: 5/50
Batch 2: 5 valid, 0 rejected. Total: 10/50
Batch 3: 5 valid, 0 rejected. Total: 15/50
Batch 4: 5 valid, 0 rejected. Total: 20/50
Batch 5: 5 valid, 0 rejected. Total: 25/50
Batch 6: 5 valid, 0 rejected. Total: 30/50
Batch 7: 5 valid, 0 rejected. Total: 35/50
Batch 8: 5 valid, 0 rejected. Total: 40/50
Batch 9: 5 valid, 0 rejected. Total: 45/50
Batch 10: 5 valid, 0 rejected. Total: 50/50

[2/10] The isolation and rejection of the outcast
API error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}. Sleeping 30s then retrying...
Batch 2: 5 valid, 0 rejected. Total: 5/50
Batch 3: 5 valid, 0 rejected. Total: 10/50
Batch 4: 5 valid, 0 rejected. Tota

### **What happened above?**

First topic was generated successfully but then Gemini API started returning 503 errors, which signify that the server is currently unavailable. While the script was still running, I tried to look for ways to deal with this but all I realized was that there isn't much I could do except wait. I realized that this was because `Gemini 3.1 Flash Lite` is a preview model and is currently unstable. I knew it was a risk from the start, but I had to go with it because of the high RPD limit.

I let the script run because even though checkpointing was working fine, the JSONL dump would happen only after all topics for the class were done. After a long wait, *Class 2* generation completed with *480 paragraphs* (20 short of the 500-paragraph target) because only 30 paragraphs could be generated for the 10th topic (i.e., only 6 calls out of the 20 attempts were successful for that topic). I decided to make up for this shortfall later using an additional script.

After manually inspecting the quality of Class 2 paragraphs, I was satisfied and hence, I typed in "yes" to move on to Class 3 generation. This is where things went south. Gemini API started returning 503 errors more frequently and the script ran for a very long time without making much progress. But since JSONL dump would happen only after all topics were done, I let it run for a long time even though I saw only a few successful calls here and there. 

The result of Class 3 generation was that only 275 records out of the required 500 were generated successfully. Both the classes were saved as JSONL files with metadata `{text, author, topic, label=1}` for Class 2 and `{text, author, topic, label=2}` for Class 3.

### **TOP-UP CELL**

In order to make up for the Gemini API 503 errors, I am going to fill whatever topics came up short due to these errors using the saved checkpoints. I will be using the `gemini-2.5-flash` model as a fallback, in case the `gemini-3.1-flash-lite-preview` model continues to be unstable. I have also changed the sleep time between API calls to 10 seconds.


I have added in-line comments in the code cell to explain the logic of the top-up script. Note that it is quite similar to the original generation script, but with a few modifications to target only the shortfall topics and to use the checkpoint data.

In [ ]:
# TOP-UP CELL

# Saving the original model and sleep time values so that we can restore them after the top-up process
original_model = MODEL_NAME
original_sleep = SLEEP_BETWEEN_CALLS

MODEL_NAME = "gemini-3.1-flash-lite-preview" # Keeping the preview model as primary
SLEEP_BETWEEN_CALLS = 10 # A more conservative approach during unstable periods

def topup_class(class_name, prompt_fn, temperature, label, output_path):
    global MODEL_NAME # I need to modify the global MODEL_NAME variable in case we need to switch to the fallback model

    rotator = KeyRotator(API_KEYS)
    new_records = []

    # Figuring out which topics are short based on the checkpoints
    shortfalls = []
    for topic_idx, topic in enumerate(TOPICS):
        saved = load_checkpoint(class_name, topic_idx)
        if len(saved) < TARGET_PER_TOPIC:
            shortfalls.append((topic_idx, topic, TARGET_PER_TOPIC - len(saved))) 

    if not shortfalls:
        print(f"{class_name}: nothing to top up")
        return

    total_needed = sum(s[2] for s in shortfalls) # Total paragraphs needed across all shortfall topics
    print(f"{class_name}: {len(shortfalls)} topics short, need {total_needed} more paragraphs")

    for topic_idx, topic, needed in shortfalls:
        print(f"\n  [{topic_idx + 1}] '{topic}' : {needed} needed") 

        # Trying the primary model first
        paragraphs = generate_for_topic(
            topic, topic_idx, class_name, prompt_fn, temperature, rotator
        )

        # If primary model exhausts all its attempts, the fallback model will try to fill the remaining gap
        if len(paragraphs) < TARGET_PER_TOPIC:
            still_short = TARGET_PER_TOPIC - len(paragraphs) # Number of paragraphs still needed after primary model attempts
            print(f"Gemini 3.1 Flash couldn't fill the gap. Switching to Gemini 2.5 Flash for {still_short} paragraphs.")
            MODEL_NAME = "gemini-2.5-flash" # Switching to the fallback model
            paragraphs = generate_for_topic(
                topic, topic_idx, class_name, prompt_fn, temperature, rotator
            )
            MODEL_NAME = "gemini-3.1-flash-lite-preview"  # Resetting to primary model for the next topic

        for para in paragraphs:
            new_records.append({
                "text": para,
                "topic": topic,
                "label": label,
                "word_count": word_count(para)
            })

    if not new_records:
        print(f"No new records generated for {class_name}")
        return

    # Appending new records to the existing JSONL file
    with open(output_path, 'a', encoding='utf-8') as f:
        for record in new_records:
            f.write(json.dumps(record) + '\n')
    print(f"\nAppended {len(new_records)} records to {output_path}")

    # Re-shuffling the entire file so the top-up records don't cluster at the end. It is important to do this because
    # the last N topic-ordered records could introduce subtle ordering bias during training in the later phases.
    with open(output_path, 'r', encoding='utf-8') as f:
        all_records = [json.loads(line) for line in f if line.strip()]

    random.seed(RANDOM_SEED)
    random.shuffle(all_records)

    with open(output_path, 'w', encoding='utf-8') as f:
        for record in all_records:
            f.write(json.dumps(record) + '\n')

    print(f"Re-shuffled the records. Final count: {len(all_records)}")

    # Checking the final per-topic counts after the top-up
    per_topic = {}
    for r in all_records:
        per_topic[r["topic"]] = per_topic.get(r["topic"], 0) + 1
    print(f"\nPer-topic final counts:")
    for topic in TOPICS:
        count = per_topic.get(topic, 0)
        flag = "" if count == TARGET_PER_TOPIC else f"  ← still short by {TARGET_PER_TOPIC - count}"
        print(f"    {count:3d}/50  {topic}{flag}")


# Running the function for both classes
print("-" * 70)
topup_class("class2", prompt_class2, TEMPERATURE_CLASS2, 1,
            "../data/class2/class2_generic_ai.jsonl")

print() # Just for better readability
topup_class("class3", prompt_class3, TEMPERATURE_CLASS3, 2,
            "../data/class3/class3_mimic_ai.jsonl")

# Restoring globals to original values (pre-top-up)
MODEL_NAME = original_model
SLEEP_BETWEEN_CALLS = original_sleep

# Final count summary
print("\n" + "-" * 70)
c2_count = sum(1 for _ in open("../data/class2/class2_generic_ai.jsonl"))
c3_count = sum(1 for _ in open("../data/class3/class3_mimic_ai.jsonl"))
print(f"Class 2: {c2_count}/500")
print(f"Class 3: {c3_count}/500")
if c2_count == 500 and c3_count == 500:
    print("All targets met.")
else:
    print("Still short on some topics. Will need to re-run this cell.")
print("-" * 70)

----------------------------------------------------------------------
Checkpoint found: 50 paragraphs already collected
Checkpoint found: 50 paragraphs already collected
Checkpoint found: 50 paragraphs already collected
Checkpoint found: 50 paragraphs already collected
Checkpoint found: 50 paragraphs already collected
Checkpoint found: 50 paragraphs already collected
Checkpoint found: 50 paragraphs already collected
Checkpoint found: 50 paragraphs already collected
Checkpoint found: 50 paragraphs already collected
Checkpoint found: 30 paragraphs already collected
class2: 1 topics short, need 20 more paragraphs

  [10] 'The past as a trap' : 20 needed
Checkpoint found: 30 paragraphs already collected
API error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}. Sleeping 30s then retrying...
API error: 503 UNAVAILABLE. {'error': {'code': 503, '

KeyboardInterrupt: 

### **What happened again!?**

The script started off by loading all the checkpoints for Class 2 data. It identified that only the 10th topic is short by 20 paragraphs of the target, and hence, it called the Gemini API. However, `gemini-3.1-flash-lite-preview` returned 503 errors in all 20 API calls, and hence, the fallback model `gemini-2.5-flash` was used, which successfully generated the required paragraphs without any errors.

But, a logic issue from my side was that the new 20 paragraphs along with the old 30 topics of the 10th topic were appended to the JSONL file, resulting in a total of 530 paragraphs. I spotted this duplicacy when the Class 3 top-up had started and around 18 API calls had already returned 503 errors. 

The fix for this logic issue is that we need to know **how many records were in the checkpoint before generation** so we can extract **only the newly generated paragraphs**. I decided to stop the script, fix the logic issue, switch to Gemini 2.5 Flash for the remaining Class 3 top-up, and run it again.

<br>

**[EDIT]:** Please refer to the in-line comments of the upcoming cells for context and rationales.

In [11]:
# I made a mistake in the previous approach. I appended the new records along with the previously cached checkpoints
# Hence, there are 530 records right now in the Class 3 JSONL file. I am fixing that in this and the next cell.

with open("../data/class2/class2_generic_ai.jsonl", 'r', encoding='utf-8') as f:
    all_records = [json.loads(line) for line in f if line.strip()]

# Counting paragraphs per topic before the fix
per_topic_before = {}
for r in all_records:
    per_topic_before[r["topic"]] = per_topic_before.get(r["topic"], 0) + 1

print("BEFORE FIX:")
for topic in TOPICS:
    count = per_topic_before.get(topic, 0)
    status = "Ok" if count == 50 else "Not Ok" # Marking the topic that has excess records as "Not Ok"
    print(f"  {status} {count:3d}/50  {topic}")

topic_10 = TOPICS[9]  # "The past as a trap" topic
topic_10_records = [r for r in all_records if r["topic"] == topic_10] # Separating out the topic 10 records from the others
other_records = [r for r in all_records if r["topic"] != topic_10] 

# Removing exact duplicates by keying on the text field. Hence, only one copy of each unique paragraph will be retained
unique_texts = {}
for record in topic_10_records:
    unique_texts[record["text"]] = record # Same text = overwrite = only one copy kept

# Converting back to list and keeping only the first 50 records
unique_topic_10 = list(unique_texts.values())[:TARGET_PER_TOPIC]

# Combining the corrected topic 10 records with the other topics' records
corrected_records = unique_topic_10 + other_records

# Shuffling to avoid topic clustering bias
random.seed(RANDOM_SEED)
random.shuffle(corrected_records)

with open("../data/class2/class2_generic_ai.jsonl", 'w', encoding='utf-8') as f:
    for record in corrected_records:
        f.write(json.dumps(record) + '\n')

# Verifying the fix
per_topic_after = {}
for r in corrected_records:
    per_topic_after[r["topic"]] = per_topic_after.get(r["topic"], 0) + 1

print("\nAFTER FIX:")
total_after = 0
for topic in TOPICS:
    count = per_topic_after.get(topic, 0)
    total_after += count
    status = "Ok" if count == 50 else "Not Ok"
    print(f"  {status} {count:3d}/50  {topic}")

print(f"\nTotal records: {total_after} (removed {len(all_records) - len(corrected_records)} duplicates)")
print("-" * 70)

BEFORE FIX:
  Ok  50/50  The hubris of scientific creation
  Ok  50/50  The isolation and rejection of the outcast
  Ok  50/50  The ethics of love, parenthood, and abandonment
  Ok  50/50  Grief and the impossibility of recovery
  Ok  50/50  Nature and the sublime as emotional mirror
  Ok  50/50  Social class and the illusion of gentility
  Ok  50/50  Wealth, ambition, and moral corruption
  Ok  50/50  Crime, guilt, and redemption
  Ok  50/50  Self-deception and identity
  Not Ok  80/50  The past as a trap

AFTER FIX:
  Ok  50/50  The hubris of scientific creation
  Ok  50/50  The isolation and rejection of the outcast
  Ok  50/50  The ethics of love, parenthood, and abandonment
  Ok  50/50  Grief and the impossibility of recovery
  Ok  50/50  Nature and the sublime as emotional mirror
  Ok  50/50  Social class and the illusion of gentility
  Ok  50/50  Wealth, ambition, and moral corruption
  Ok  50/50  Crime, guilt, and redemption
  Ok  50/50  Self-deception and identity
  Ok  50/50 

### **Now what?**

Now that Class 2 dataset is complete, I am going to make the `Gemini 2.5 Flash` my primary model for API calls. I will use the modified JSONL dumping logic for only saving the newly generated paragraphs to the JSONL file. I will also use an increased sleep time of 10 seconds between API calls to reduce the chances of hitting the requests per minute (RPM) limit.

In [ ]:

# I am shifting from Gemini 3.1 Flash Lite Preview to Gemini 2.5 Flash because it can be seen in the previous cell's logs that the former is consistently giving 503 errors
# I was trying to maintain consistency by using Flash 3.1 Lite Previiew in the previous cell but I can't anymore, it is annoying to keep being bombed by the server errors.


# Using Gemini 2.5 Flash
MODEL_NAME = "gemini-2.5-flash"
SLEEP_BETWEEN_CALLS = 10  # Conservative sleep during recovery period

def topup_class(class_name, prompt_fn, temperature, label, output_path):
    
    rotator = KeyRotator(API_KEYS)
    new_records = []

    # Figuring out which topics are short based on the checkpoints
    shortfalls = []
    for topic_idx, topic in enumerate(TOPICS):
        saved = load_checkpoint(class_name, topic_idx)
        if len(saved) < TARGET_PER_TOPIC:
            shortfalls.append((topic_idx, topic, TARGET_PER_TOPIC - len(saved))) 

    if not shortfalls:
        print(f"{class_name}: nothing to top up")
        return

    total_needed = sum(s[2] for s in shortfalls)  # Total paragraphs needed across all shortfall topics
    print(f"{class_name}: {len(shortfalls)} topics short, need {total_needed} more paragraphs")

    for topic_idx, topic, needed in shortfalls:
        print(f"\n  [{topic_idx + 1}] '{topic}' : {needed} needed") 

        # Tracking checkpoint size BEFORE generation to identify only the new paragraphs (generate_for_topic returns full 50-record list, but includes cached checkpoint records)
        prior_checkpoint = load_checkpoint(class_name, topic_idx)
        prior_count = len(prior_checkpoint)

        paragraphs = generate_for_topic(
            topic, topic_idx, class_name, prompt_fn, temperature, rotator
        )

        # Only appending the new paragraphs generated in this batch
        new_only = paragraphs[prior_count:]

        for para in new_only:
            new_records.append({
                "text": para,
                "topic": topic,
                "label": label,
                "word_count": word_count(para)
            })

    if not new_records:
        print(f"No new records generated for {class_name}")
        return

    # Appending new records to the existing JSONL file
    with open(output_path, 'a', encoding='utf-8') as f:
        for record in new_records:
            f.write(json.dumps(record) + '\n')
    print(f"\nAppended {len(new_records)} records to {output_path}")

    # Re-shuffling the entire file so the top-up records don't cluster at the end
    # This avoids introducing topic-wise ordering bias during training later on
    with open(output_path, 'r', encoding='utf-8') as f:
        all_records = [json.loads(line) for line in f if line.strip()]

    random.seed(RANDOM_SEED)
    random.shuffle(all_records)

    with open(output_path, 'w', encoding='utf-8') as f:
        for record in all_records:
            f.write(json.dumps(record) + '\n')

    print(f"Re-shuffled the records. Final count: {len(all_records)}")

    # Checking the final per-topic counts after the top-up
    per_topic = {}
    for r in all_records:
        per_topic[r["topic"]] = per_topic.get(r["topic"], 0) + 1
    print(f"\nPer-topic final counts:")
    for topic in TOPICS:
        count = per_topic.get(topic, 0)
        flag = "" if count == TARGET_PER_TOPIC else f"  ← still short by {TARGET_PER_TOPIC - count}"
        print(f"    {count:3d}/50  {topic}{flag}")


# Running the function for Class 3 only
print("-" * 70)
topup_class("class3", prompt_class3, TEMPERATURE_CLASS3, 2,
            "../data/class3/class3_mimic_ai.jsonl")

# Final count summary
print("\n" + "-" * 70)
c3_count = sum(1 for _ in open("../data/class3/class3_mimic_ai.jsonl"))
print(f"Class 3: {c3_count}/500")
if c3_count == 500:
    print("All targets met.")
else:
    print("Still short on some topics. Will need to re-run this cell.")
print("-" * 70)

----------------------------------------------------------------------
Checkpoint found: 50 paragraphs already collected
Checkpoint found: 35 paragraphs already collected
Checkpoint found: 50 paragraphs already collected
Checkpoint found: 50 paragraphs already collected
Checkpoint found: 10 paragraphs already collected
Checkpoint found: 35 paragraphs already collected
Checkpoint found: 15 paragraphs already collected
Checkpoint found: 15 paragraphs already collected
Checkpoint found: 15 paragraphs already collected
class3: 7 topics short, need 225 more paragraphs

  [2] 'The isolation and rejection of the outcast' : 15 needed
Checkpoint found: 35 paragraphs already collected
Checkpoint found: 35 paragraphs already collected
Batch 1: 3 valid, 2 rejected. Total: 38/50
Rejected word counts: [125, 129]
Batch 2: 5 valid, 0 rejected. Total: 43/50
Batch 3: 5 valid, 0 rejected. Total: 48/50
Batch 4: 4 valid, 1 rejected. Total: 50/50
Rejected word counts: [204]

  [5] 'Nature and the sublime as

### **Some important observations from above**

- For the Class 3 top-up, I saw a significant improvement in the success rate of API calls after switching to `Gemini 2.5 Flash`. Also, since its RPD limit is 20, all my API keys had to be rotated through the top-up process. I am glad the API key rotation logic worked fine.

- I also observed that the `Gemini 2.5 Flash` often failed to follow the word count instructions in the prompt. My word count filtering logic was able to catch these and discard the paragraphs. A good display of the importance of such filtering logic when working with generative models. 

<br>

I now have shuffled 500 paragraphs each for both Class 2 and Class 3 datasets. I will be displaying a sample on the same topic from both classes for a quick qualitative comparison. 

In [16]:
# Displaying a sample on the same topic from both Class 2 and Class 3

def load_jsonl(filepath):
    records = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

class2_data = load_jsonl('../data/class2/class2_generic_ai.jsonl')
class3_data = load_jsonl('../data/class3/class3_mimic_ai.jsonl')
topic_of_interest = TOPICS[7] # Arbitrary choice

print(f"\nSample paragraphs on topic: '{topic_of_interest}'")
print("-"*70)

# Using next() with a generator expression to find the first record matching the topic of interest in each class. If no record is found, it will return None
sample_c2 = next((r for r in class2_data if r["topic"] == topic_of_interest), None)
sample_c3 = next((r for r in class3_data if r["topic"] == topic_of_interest), None)
if sample_c2:
    print(f"\n[Class 2 - Generic AI]")
    print(sample_c2['text'][:700] + "..." if len(sample_c2['text']) > 700 else sample_c2['text'])

if sample_c3:
    print(f"\n[Class 3 - Mimic AI]")
    print(sample_c3['text'][:700] + "..." if len(sample_c3['text']) > 700 else sample_c3['text'])

print("\n" + "-"*70)



Sample paragraphs on topic: 'Crime, guilt, and redemption'
----------------------------------------------------------------------

[Class 2 - Generic AI]
The heavy iron door clicked shut, leaving behind the chaotic noise of the city streets for a silent, sterile corridor. A man stood frozen in the dim light, tracing the jagged scar on his palm that served as a permanent record of a reckless night. He had taken what did not belong to him, driven by a desperate need that blurred the lines between necessity and greed. Now, the weight of his actions pressed against his chest like a physical burden, making every breath feel deliberate and heavy. Outside, life continued without him, indifferent to the single choice that had fractured his existence. He stared at the whitewashed walls, realizing that the cold reality of his confinement was merely the...

[Class 3 - Mimic AI]
In those moments when the ceaseless tempest of my internal anguish threatened to overwhelm the very vessel of my reason

### **Final Steps**

Finally, I will be splitting the combined dataset of Class 1, 2, and 3 into train/val/test splits in the ratio of 70/15/15, stratified by label. I will be using `sklearn.model_selection.train_test_split` with `stratify=y` to achieve this. Stratification will ensure that each split (train/val/test) has the same proportion of each class as in the original dataset.

Since my dataset is 33.3% Class 1, 33.3% Class 2 and 33.3% Class 3, each split will maintain 33.3% of each class.
Without stratification, I might randomly end up with 50% Class 1 in train adn 15% in test.

An important first step to this is **downsampling Class 1 from 1000 paragraphs to 500 paragraphs** so that it is balanced with Class 2 and Class 3. I will be doing this random sampling with a fixed random seed (42) for reproducibility.

If I don't do this, my **models in the future will see twice as many human examples as AI examples during training**, which will bias them towards predicting "human".

In [ ]:
from sklearn.model_selection import train_test_split

# A helper function for loading JSONL files into a list of dictionaries. 
def load_jsonl(filepath):
    records = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

class1_shelley = load_jsonl('../data/class1/class1_shelley.jsonl')
class1_dickens = load_jsonl('../data/class1/class1_dickens.jsonl')
class2_records = load_jsonl('../data/class2/class2_generic_ai.jsonl')
class3_records = load_jsonl('../data/class3/class3_mimic_ai.jsonl')

print(f"Loaded — Shelley: {len(class1_shelley)}, Dickens: {len(class1_dickens)}, "
      f"Class2: {len(class2_records)}, Class3: {len(class3_records)}")

# Downsampling Class 1 to 500 records (250 from each author)
random.seed(RANDOM_SEED)
class1_records = random.sample(class1_shelley, 250) + random.sample(class1_dickens, 250)

print(f"\nAfter downsampling — Class 1: {len(class1_records)}, "
      f"Class 2: {len(class2_records)}, Class 3: {len(class3_records)}")


# Merging all classes and shuffling to avoid any ordering bias in the later splits (train/val/test)
all_records = class1_records + class2_records + class3_records
random.seed(RANDOM_SEED)
random.shuffle(all_records) 

total = len(all_records)
print(f"Total records: {total}")

# Stratifying splitting by label: 70% train / 15% val / 15% test
# I will use a two-step split: carving out 15% test, then splitting the remainder (which gives 15% of total as val and 70% as train).

labels = [r['label'] for r in all_records] # All the labels in the dataset 

train_val, test = train_test_split(
    all_records,
    test_size=0.15, 
    stratify=labels,
    random_state=RANDOM_SEED
)

# Labels of the subset used for train/val split. This will ensure that stratification is maintained in the second split as well
train_val_labels = [r['label'] for r in train_val] 

train, val = train_test_split(
    train_val,
    test_size=(0.15 / 0.85),  # 15% of the total
    stratify=train_val_labels,
    random_state=RANDOM_SEED
)

print(f"\nSplit sizes:")
print(f"  Train: {len(train):4d} ({len(train)/total*100:.1f}%)")
print(f"  Val:   {len(val):4d} ({len(val)/total*100:.1f}%)")
print(f"  Test:  {len(test):4d} ({len(test)/total*100:.1f}%)")

# To verify if stratification worked, class proportions should be ~33% each across all splits (as discussed in the MD cell above)
def check_stratification(records, split_name): 
    print(f"\n  {split_name} ({len(records)} records):")
    for label in [0, 1, 2]: # Counting the number of records for each label in the given split
        count = sum(1 for r in records if r['label'] == label)
        print(f"    Class {label}: {count} ({count/len(records)*100:.1f}%)")

print("\nStratification check:")
check_stratification(all_records, "Full dataset")
check_stratification(train, "Train")
check_stratification(val, "Val")
check_stratification(test, "Test")

# Saving the splits as 3 separate JSONL files in the 'splits' subdirectory of the 'data' directory and the full shuffled dataset as well in the 'data' directory
for split_name, split_data in [('train', train), ('val', val), ('test', test)]:
    path = f'../data/splits/{split_name}.jsonl'
    with open(path, 'w', encoding='utf-8') as f:
        for record in split_data:
            f.write(json.dumps(record) + '\n')
    print(f"Saved {split_name}: {len(split_data)} records → {path}")

with open('../data/full_dataset.jsonl', 'w', encoding='utf-8') as f:
    for record in all_records:
        f.write(json.dumps(record) + '\n')
print(f"Saved full dataset: {len(all_records)} records → ../data/splits/full_dataset.jsonl")

Loaded — Shelley: 500, Dickens: 500, Class2: 500, Class3: 500

After downsampling — Class 1: 500, Class 2: 500, Class 3: 500
Total records: 1500

Split sizes:
  Train: 1049 (69.9%)
  Val:    226 (15.1%)
  Test:   225 (15.0%)

Stratification check:

  Full dataset (1500 records):
    Class 0: 500 (33.3%)
    Class 1: 500 (33.3%)
    Class 2: 500 (33.3%)

  Train (1049 records):
    Class 0: 350 (33.4%)
    Class 1: 350 (33.4%)
    Class 2: 349 (33.3%)

  Val (226 records):
    Class 0: 75 (33.2%)
    Class 1: 75 (33.2%)
    Class 2: 76 (33.6%)

  Test (225 records):
    Class 0: 75 (33.3%)
    Class 1: 75 (33.3%)
    Class 2: 75 (33.3%)
Saved train: 1049 records → ../data/train.jsonl
Saved val: 226 records → ../data/val.jsonl
Saved test: 225 records → ../data/test.jsonl
Saved full dataset: 1500 records → ../data/full_dataset.jsonl


**[[EDIT]]:** I have made a minor change in the directory structure for cleanliness. I have created a `splits` subdirectory inside the `data` directory where I have moved the train/val/test JSONL files. However, since the above code randomly shuffles the data before splitting, I can't re-run the code cell. Therefore, the log statement in the previous output has a minor path inconsistency: it says `../data/train.jsonl` instead of `../data/splits/train.jsonl`. 

### **CONCLUSION**

The dataset `../data/full_dataset.jsonl` has 1500 paragraphs in total, with 500 paragraphs each for Class 1 (human-written), Class 2 (AI-generated with no specific style), and Class 3 (AI-generated in the style of Mary Shelley). 

The train/val/test split is 70/15/15, stratified by class label, with a random seed of 42 for reproducibility. 

This means that each split will have 33.3% of each class, ensuring that the distribution of classes is consistent across train, validation, and test sets.

Splits stored at `../data/splits/train.jsonl`, `../data/splits/val.jsonl`, and `../data/splits/test.jsonl` respectively. 

## **Data Limitations**

Class 2 (Generic AI) contains 480 paragraphs from `gemini-3.1-flash-lite-preview` and 20 from `gemini-2.5-flash` due to API unavailability during generation. Class 3 (Mimic AI) contains approximately 275 paragraphs from `gemini-3.1-flash-lite-preview` and 225 from `gemini-2.5-flash` (~45% of the class).

This mixed-model composition is a limitation, therefore, any observed style differences between classes may reflect inter-model variation. This is more relevant for Class 3 because almost half the data comes from a different model. 

A cleaner experiment would have used a single model for all generations. This was not possible because Google API returned consistent server-side errors during the generation process.

I will interpret results involving Class 3 stylometric features with this caveat in mind.

### **Some citations**

Github Copilot came to clutch at times of distress, for example, in writing the API key rotation logic and the checkpointing logic. However, I had to modify the code at almost all steps because turns out, the free-tier Copilot suggestions are not satisfactory for demanding assignments like this one. But it was a good starting point and a time-saver for some of the boilerplate code.

Also, the sources I used for understanding the writing styles of the authors in the particular novels and the themes of the novels are cited in the [Topic Extraction](##topic-extraction) MD cell above.

- https://github.com/google-gemini/cookbook

- https://ai.google.dev/gemini-api/docs/models/gemini-3.1-flash-lite-preview

- https://github.com/explosion/spaCy/discussions/10895

- https://github.com/scikit-learn/scikit-learn/blob/main/sklearn/model_selection/tests/test_split.py

- https://stackoverflow.com/questions/56470403/spacy-nlp-spacy-loaden-core-web-lg

- https://spacy.io/models/en#en_core_web_md

- https://www.gutenberg.org
